# 🐾 Agente RAG - Pink Panther Knowledge Base

Este notebook implementa un **agente RAG (Retrieval-Augmented Generation)** usando:

- **smolagents**: Framework de agentes de Hugging Face
- **ChromaDB**: Base de datos vectorial para búsqueda semántica
- **sentence-transformers**: Para generar embeddings de texto
- **Amazon Bedrock (Claude)**: Como modelo de lenguaje vía LiteLLM

## Arquitectura

```
[Documentos .md] → [Chunking] → [Embeddings] → [ChromaDB]
                                                     ↓
[Pregunta usuario] → [Agente] → [Busca en ChromaDB] → [Contexto relevante]
                                                     ↓
                                      [LLM genera respuesta informada]
```

## 1. Instalación de dependencias

In [1]:
!uv pip install -q smolagents litellm chromadb sentence-transformers boto3

## 2. Descargar la Knowledge Base desde Kaggle

Dataset: https://www.kaggle.com/datasets/gustavodelacruztovar/historias-pink-panther



In [2]:
# Descargar y descomprimir el dataset
!kaggle datasets download -d gustavodelacruztovar/historias-pink-panther
!unzip -o historias-pink-panther.zip -d kbpink
!ls kbpink/

Dataset URL: https://www.kaggle.com/datasets/gustavodelacruztovar/historias-pink-panther
License(s): apache-2.0
100% 50.7k/50.7k [00:00<00:00, 1.27MB/s]

Archive:  historias-pink-panther.zip
  inflating: kbpink/01.md            
  inflating: kbpink/02.md            
  inflating: kbpink/03.md            
  inflating: kbpink/04.md            
  inflating: kbpink/05.md            
  inflating: kbpink/06.md            
  inflating: kbpink/07.md            
  inflating: kbpink/08.md            
  inflating: kbpink/09.md            
  inflating: kbpink/10.md            
  inflating: kbpink/11.md            
  inflating: kbpink/12.md            
  inflating: kbpink/13.md            
  inflating: kbpink/elprincipito.md  
01.md  03.md  05.md  07.md  09.md  11.md  13.md
02.md  04.md  06.md  08.md  10.md  12.md  elprincipito.md


## 3. Configurar Amazon Bedrock

El secreto `AWS_BEARER_TOKEN_BEDROCK` se puede guardar en Colab Secrets (🔑 panel izquierdo) o pegarlo directamente.

In [3]:
import os

# Opción 1: Usar Colab Secrets (recomendado)
try:
    from google.colab import userdata
    os.environ["AWS_BEARER_TOKEN_BEDROCK"] = userdata.get("AWS_BEARER_TOKEN_BEDROCK")
    print("✅ Token cargado desde Colab Secrets")
except Exception:
    # Opción 2: Pegar directamente
    os.environ["AWS_BEARER_TOKEN_BEDROCK"] = "PEGA_TU_TOKEN_AQUI"  # <-- Cambia esto
    print("⚠️ Token configurado manualmente")

os.environ["AWS_REGION_NAME"] = "us-east-1"

✅ Token cargado desde Colab Secrets


In [4]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="bedrock/converse/us.anthropic.claude-sonnet-4-20250514-v1:0",
    aws_region_name="us-east-1",
)
print("✅ Modelo Bedrock (Claude) configurado")

✅ Modelo Bedrock (Claude) configurado


## 4. Cargar y procesar documentos

In [5]:
import glob

DOCS_DIR = "kbpink"

def load_markdown_documents(directory: str) -> list[dict]:
    """Carga todos los archivos .md del directorio, excluyendo 'elprincipito.md'."""
    documents = []
    md_files = sorted(glob.glob(os.path.join(directory, "**", "*.md"), recursive=True))
    excluded_file = os.path.join(directory, "elprincipito.md") # Define the full path of the file to exclude
    for filepath in md_files:
        if filepath == excluded_file: # Skip the excluded file
            continue
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        filename = os.path.basename(filepath)
        documents.append({"content": content, "filename": filename, "filepath": filepath})
    print(f"📄 Cargados {len(documents)} documentos desde {directory} (excluyendo 'elprincipito.md')")
    return documents


def chunk_document(doc: dict, chunk_size: int = 1000, overlap: int = 200) -> list[dict]:
    """Divide un documento en chunks con overlap."""
    text = doc["content"]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]
        if end < len(text):
            last_newline = chunk_text.rfind("\n\n")
            if last_newline > chunk_size // 2:
                chunk_text = chunk_text[:last_newline]
                end = start + last_newline
        chunks.append({
            "content": chunk_text.strip(),
            "filename": doc["filename"],
            "chunk_id": f"{doc['filename']}_chunk_{len(chunks)}",
        })
        start = end - overlap if end < len(text) else len(text)
    return chunks

documents = load_markdown_documents(DOCS_DIR)

📄 Cargados 13 documentos desde kbpink (excluyendo 'elprincipito.md')


## 5. Construir la base de datos vectorial (ChromaDB)

In [6]:
import chromadb
from sentence_transformers import SentenceTransformer

# Chunking
all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_document(doc))
print(f"🔪 Generados {len(all_chunks)} chunks")



🔪 Generados 68 chunks


In [7]:
# Embeddings
print("🧠 Cargando modelo de embeddings...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [chunk["content"] for chunk in all_chunks]
print("📐 Generando embeddings...")
embeddings = embedding_model.encode(texts, show_progress_bar=True)

🧠 Cargando modelo de embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📐 Generando embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [8]:
# ChromaDB
client = chromadb.Client()
collection = client.get_or_create_collection(name="pink_panther_kb")
collection.add(
    ids=[chunk["chunk_id"] for chunk in all_chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"filename": chunk["filename"]} for chunk in all_chunks],
)
print(f"✅ Base vectorial creada con {collection.count()} documentos")

✅ Base vectorial creada con 68 documentos


## 6. Crear la herramienta de Retrieval y el Agente

In [9]:
from smolagents import Tool, CodeAgent


class RAGRetrieverTool(Tool):
    """Herramienta de búsqueda semántica en la KB de Pink Panther."""

    name = "knowledge_base_search"
    description = (
        "Busca información relevante en la base de conocimiento de historias de Pink Panther. "
        "Usa esta herramienta para encontrar fragmentos de texto relacionados con tu consulta."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "La consulta de búsqueda en lenguaje natural.",
        }
    }
    output_type = "string"

    def __init__(self, collection, embedding_model, **kwargs):
        super().__init__(**kwargs)
        self.collection = collection
        self.embedding_model = embedding_model

    def forward(self, query: str) -> str:
        query_embedding = self.embedding_model.encode([query]).tolist()
        results = self.collection.query(query_embeddings=query_embedding, n_results=5)
        if not results["documents"][0]:
            return "No se encontraron documentos relevantes."
        output = "\n📚 Documentos recuperados:\n"
        for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
            output += f"\n{'='*60}\n📄 Doc {i+1} ({meta['filename']})\n{'='*60}\n{doc}\n"
        return output


retriever_tool = RAGRetrieverTool(collection, embedding_model)

agent = CodeAgent(
    tools=[retriever_tool],
    model=model,
    max_steps=4,
    verbosity_level=2,
)
print("✅ Agente RAG listo")

✅ Agente RAG listo


## 7. Probar el Agente

In [10]:
result = agent.run("¿Qué reflexiones hace Pink Panther sobre la inteligencia artificial?")
print("\n💡 RESPUESTA:")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ¿Qué reflexiones hace Pink Panther sobre la inteligencia artificial?                                            │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/converse/us.anthropic.claude-sonnet-4-20250514-v1:0 ────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to search for information about Pink Panther's reflections on artificial intelligence. I'll use the
knowledge base search tool to find relevant content about this topic.                                              
                                                                                                                   
<code>                                                                                                             
result = knowledge_base_search("Pink Panther inteligencia artificial reflexiones")                                 
print(result)                                                                                                      
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = knowledge_base_search("Pink Panther inteligencia artificial reflexiones")                               
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:

📚 Documentos recuperados:

============================================================
📄 Doc 1 (06.md)
============================================================
onario, mostró que el universo no era solo un tejido plano, como lo describían las ecuaciones de Einstein, sino una
estructura en espiral que permitía la interacción de fuerzas opuestas. Las curvas de esta espiral conectaban la 
gravedad con las otras fuerzas mediante una interacción inversamente proporcional. Pink Panther resumió su idea en 
una ecuación que describía cómo los gravitones y los bosones portadores de las otras fuerzas intercambiaban energía
en ciclos opuestos.
El Experimento Final
Pink Panther decidió probar su teoría con un dispositivo que él mismo diseñó: el Resonador de Fuerzas Cósmicas. 
Este dispositivo consistía en un campo de energía generado por partículas opuestas que simulaban los efectos de la 
gravedad y las fuerzas de GUT. Activarlo requería una sincronización perfecta, y cualquier error podría colapsar el
tejido del espacio-tiempo.
Con un profundo respiro, Pink Panther activó el resonador. Durante unos momentos, el laboratorio quedó envuelto en 
un campo de ene

============================================================
📄 Doc 2 (11.md)
============================================================
ión los semáforos rojos y verdes, los Procesos trabajaban sin detenerse, aprendiendo sobre la importancia de la 
sincronización y los mecanismos de comunicación.
Un momento revelador
En medio de la actividad, un estudiante levantó la cabeza y dijo: “¡Ahora entiendo! La memoria no solo es un lugar 
para guardar datos, sino que también tiene que ser gestionada como un recurso compartido.” La Pink Panther asintió 
con una sonrisa. El objetivo de la actividad estaba cumpliéndose: en lugar de escuchar una explicación abstracta, 
los estudiantes vivían la experiencia, comprendiendo conceptos complejos con una claridad inesperada.
Inspiración alienígena
Al final de la sesión, mientras los estudiantes reflexionaban sobre lo aprendido, la Pink Panther compartió de 
dónde había surgido la idea. Les habló de un fragmento de El problema de los tres cuerpos, un libro donde una 
civilización extraterrestre construía una computadora digital utilizando seres vivos. Aunque el contexto era 
diferente, la Pink

============================================================
📄 Doc 3 (06.md)
============================================================
La Gran Unificación de Pink Panther
En un rincón secreto del cosmos, oculto bajo la superficie de un asteroide recubierto de cristales fluorescentes, 
Pink Panther trabajaba en su laboratorio estelar. Era un lugar donde lo imposible se hacía probable y las leyes de 
la física eran apenas puntos de partida para su mente creativa. Su obsesión: unificar la gravedad con la Gran 
Teoría Unificada (GUT) que abarcaba las otras tres fuerzas fundamentales: electromagnetismo, interacción débil e 
interacción fuerte.
A lo largo de los siglos, físicos eminentes habían intentado, sin éxito, resolver este enigma. Pero Pink Panther, 
con su característico ingenio y audacia, estaba convencido de que la clave residía en algo que todos habían pasado 
por alto: las fuerzas opuestas que danzan en el tejido del universo.
El Momento del Descubrimiento
Una noche, mientras manipulaba partículas en su acelerador cuántico personalizado, un destello brillante iluminó su
laboratorio. Era el resultado inesperado de un e

============================================================
📄 Doc 4 (03.md)
============================================================
Pink Panther caminó lentamente por la Alameda, el atardecer de la ciudad vistiéndose de oro. Sonreía. No era una 
gran compositora como Elgar, pero había tejido sus propios enigmas: amistades profundas, días de lucha, logros que 
sólo ella conocía. Y sobre todo, había amado.
Y esa noche, como pocas veces, se sintió en paz.

=======================================================

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: Based on the search results, I now have enough information to provide a comprehensive answer about Pink   
Panther's reflections on artificial intelligence. Let me compile the key insights from the retrieved documents.    
                                                                                                                   
<code>                                                                                                             
final_answer("""Basándome en la información encontrada, Pink Panther hace varias reflexiones profundas sobre la    
inteligencia artificial:                                                                                           
                                                                                                                   
**1. Crítica al paradigma dominante de automatización:**                                                           
Pink Panther observa que la inteligencia artificial avanza con velocidad extraordinaria, pero el discurso dominante
gira alrededor de una sola idea: "automatizar todo" - automatizar el trabajo, automatizar el conocimiento, reducir 
la necesidad de humanos. Ella percibe que el conocimiento mismo está entrando a una especie de "industrialización  
masiva".                                                                                                           
                                                                                                                   
**2. El concepto de "colonialismo de datos":**                                                                     
Pink Panther reflexiona sobre una frase que escuchó: "Esto empieza a parecer colonialismo de datos", sugiriendo que
existe una forma de explotación en cómo se maneja la información y los datos en la era de la IA.                   
                                                                                                                   
**3. Propuesta de un enfoque alternativo:**                                                                        
En lugar de competir con la inteligencia artificial, Pink Panther propone pensar en "cómo puede evolucionar la     
humanidad". Sugiere que es momento de imaginar:                                                                    
- Otras formas de sociedad                                                                                         
- Otras filosofías                                                                                                 
- Otras maneras de entender al ser humano                                                                          
                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("""Basándome en la información encontrada, Pink Panther hace varias reflexiones profundas sobre la  
  inteligencia artificial:                                                                                         
                                                                                                                   
  **1. Crítica al paradigma dominante de automatización:**                                                         
  Pink Panther observa que la inteligencia artificial avanza con velocidad extraordinaria, pero el discurso        
  dominante gira alrededor de una sola idea: "automatizar todo" - automatizar el trabajo, automatizar el           
  conocimiento, reducir la necesidad de humanos. Ella percibe que el conocimiento mismo está entrando a una        
  especie de "industrialización masiva".                                                                           
                                                                                                                   
  **2. El concepto de "colonialismo de datos":**                                                                   
  Pink Panther reflexiona sobre una frase que escuchó: "Esto empieza a parecer colonialismo de datos", sugiriendo  
  que existe una forma de explotación en cómo se maneja la información y los datos en la era de la IA.             
                                                                                                                   
  **3. Propuesta de un enfoque alternativo:**                                                                      
  En lugar de competir con la inteligencia artificial, Pink Panther propone pensar en "cómo puede evolucionar la   
  humanidad". Sugiere que es momento de imaginar:                                                                  
  - Otras formas de sociedad                                                                                       
  - Otras filosofías                                                                                               
  - Otras maneras de entender al ser humano                                                                        
                                                                                                                   
  **4. Inspiración en la ciencia ficción:**                                                                        
  Pink Panther se inspira en historias de ciencia ficción que no ponen la inteligencia artificial en el centro,    
  sino que se enfocan en "la evolución de la mente humana" y en "humanos expandiendo su conciencia", como las      
  obras de Asimov, Herbert y Dan Simmons.                                                                          
                                                                                                                   
  **5. Cuestionamiento del sistema económico:**                                                                    
  Reflexiona sobre cómo el sistema económico está empezando a tratar el conocimiento como algo desechable, lo que  
  la lleva a plantear la necesidad de nuevas formas de organización social.                                        
                                                                                                                   
  En esencia, Pink Panther ve la IA no como una amenaza a combatir, sino como una oportunidad para repensar        
  fundamentalmente la evolución humana y social.""")                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Basándome en la información encontrada, Pink Panther hace varias reflexiones profundas sobre la 
inteligencia artificial:

**1. Crítica al paradigma dominante de automatización:**
Pink Panther observa que la inteligencia artificial avanza con velocidad extraordinaria, pero el discurso dominante
gira alrededor de una sola idea: "automatizar todo" - automatizar el trabajo, automatizar el conocimiento, reducir 
la necesidad de humanos. Ella percibe que el conocimiento mismo está entrando a una especie de "industrialización 
masiva".

**2. El concepto de "colonialismo de datos":**
Pink Panther reflexiona sobre una frase que escuchó: "Esto empieza a parecer colonialismo de datos", sugiriendo que
existe una forma de explotación en cómo se maneja la información y los datos en la era de la IA.

**3. Propuesta de un enfoque alternativo:**
En lugar de competir con la inteligencia artificial, Pink Panther propone pensar en "cómo puede evolucionar la 
humanidad". Sugiere que es momento de imaginar:
- Otras formas de sociedad
- Otras filosofías  
- Otras maneras de entender al ser humano

**4. Inspiración en la ciencia ficción:**
Pink Panther se inspira en historias de ciencia ficción que no ponen la inteligencia artificial en el centro, sino 
que se enfocan en "la evolución de la mente humana" y en "humanos expandiendo su conciencia", como las obras de 
Asimov, Herbert y Dan Simmons.

**5. Cuestionamiento del sistema económico:**
Reflexiona sobre cómo el sistema económico está empezando a tratar el conocimiento como algo desechable, lo que la 
lleva a plantear la necesidad de nuevas formas de organización social.

En esencia, Pink Panther ve la IA no como una amenaza a combatir, sino como una oportunidad para repensar 
fundamentalmente la evolución humana y social.

[Step 4: Duration 8.92 seconds| Input tokens: 18,589 | Output tokens: 774]


💡 RESPUESTA:
Basándome en la información encontrada, Pink Panther hace varias reflexiones profundas sobre la inteligencia artificial:

**1. Crítica al paradigma dominante de automatización:**
Pink Panther observa que la inteligencia artificial avanza con velocidad extraordinaria, pero el discurso dominante gira alrededor de una sola idea: "automatizar todo" - automatizar el trabajo, automatizar el conocimiento, reducir la necesidad de humanos. Ella percibe que el conocimiento mismo está entrando a una especie de "industrialización masiva".

**2. El concepto de "colonialismo de datos":**
Pink Panther reflexiona sobre una frase que escuchó: "Esto empieza a parecer colonialismo de datos", sugiriendo que existe una forma de explotación en cómo se maneja la información y los datos en la era de la IA.

**3. Propuesta de un enfoque alternativo:**
En lugar de competir con la inteligencia artificial, Pink Panther propone pensar en "cómo puede evolucionar la humanidad". Sugiere que es momento d

In [11]:
result = agent.run("¿Cuál es la relación entre Pink Panther y Gus?")
print("\n💡 RESPUESTA:")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ¿Cuál es la relación entre Pink Panther y Gus?                                                                  │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/converse/us.anthropic.claude-sonnet-4-20250514-v1:0 ────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to search for information about the relationship between Pink Panther and Gus in the knowledge     
base. Let me start by searching for information about both characters and their connection.                        
                                                                                                                   
<code>                                                                                                             
result = knowledge_base_search("Pink Panther Gus relación")                                                        
print(result)                                                                                                      
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = knowledge_base_search("Pink Panther Gus relación")                                                      
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:

📚 Documentos recuperados:

============================================================
📄 Doc 1 (03.md)
============================================================
Pink Panther caminó lentamente por la Alameda, el atardecer de la ciudad vistiéndose de oro. Sonreía. No era una 
gran compositora como Elgar, pero había tejido sus propios enigmas: amistades profundas, días de lucha, logros que 
sólo ella conocía. Y sobre todo, había amado.
Y esa noche, como pocas veces, se sintió en paz.

============================================================
📄 Doc 2 (06.md)
============================================================
loración espacial, la energía y la comprensión de la realidad misma. Aunque el mundo lo celebró como un genio 
incomparable, Pink Panther, fiel a su naturaleza modesta, simplemente sonrió y dijo:
“La respuesta siempre estuvo allí, bailando en el cosmos. Solo hacía falta verla desde el ángulo correcto.”
Y así, Pink Panther pasó a la historia no solo como un maestro de la enseñanza y el misterio, sino también como el 
genio que unificó el universo.

============================================================
📄 Doc 3 (06.md)
============================================================
La Gran Unificación de Pink Panther
En un rincón secreto del cosmos, oculto bajo la superficie de un asteroide recubierto de cristales fluorescentes, 
Pink Panther trabajaba en su laboratorio estelar. Era un lugar donde lo imposible se hacía probable y las leyes de 
la física eran apenas puntos de partida para su mente creativa. Su obsesión: unificar la gravedad con la Gran 
Teoría Unificada (GUT) que abarcaba las otras tres fuerzas fundamentales: electromagnetismo, interacción débil e 
interacción fuerte.
A lo largo de los siglos, físicos eminentes habían intentado, sin éxito, resolver este enigma. Pero Pink Panther, 
con su característico ingenio y audacia, estaba convencido de que la clave residía en algo que todos habían pasado 
por alto: las fuerzas opuestas que danzan en el tejido del universo.
El Momento del Descubrimiento
Una noche, mientras manipulaba partículas en su acelerador cuántico personalizado, un destello brillante iluminó su
laboratorio. Era el resultado inesperado de un e

============================================================
📄 Doc 4 (07.md)
============================================================
La noche del sábado 8 de febrero de 2025, Pink Panther caminaba con su característico andar elegante hacia el 
majestuoso auditorio donde se llevaría a cabo un concierto que había esperado por años. El aire nocturno llevaba 
consigo la emoción de quienes sabían que algo extraordinario estaba por suceder. Pero para él, esta no era solo una
velada musical más; era un encuentro con la esencia misma de su existencia.
El programa de la noche anunciaba el Tercer Concierto para Piano de Prokofiev, interpretado por un artista que Pink
Panther había descubierto años atrás: Alexander Malofeev. No era cualquier pianista, sino aquel cuyas 
interpretaciones habían sido un faro en los días más oscuros de su vida.
Recuerdos de un Tiempo Difícil
En 2019, Pink Panther navegaba por la inmensidad de YouTube cuando, casi por accidente, se topó con un video de 
Malofeev. Aquel joven, de expresión tranquila pero dedos electrizantes, interpretaba con una intensidad casi irreal
el Tercer Concierto de Rachmaninov.

============================================================
📄 Doc 5 (08.md)
============================================================
Era una tarde nublada en la Ciudad de México cuando la Pink Panther, algo fatigada por la rutina de la semana, 
decidió hacer algo diferente y asistir a un concierto en la Sala Nezahualcóyotl. La Orquesta Filarmónica de la 
UNAM, bajo la batuta del maestro Enrique Arturo Diemecke, interpretaría "Así hablaba Zaratustra" de Richard 
Strauss. A la Pink Panther siempre le había intrigado la obra, especialmente su conexión con Nietzsche y su 
filosofía. Aunque conocía el tem

...

Execution logs:

📚 Documentos recuperados:

============================================================
📄 Doc 1 (01.md)
============================================================
inventó la Psicohistoria.

Pero como ocurre con muchos lectores de ciencia ficción, aquella imaginación había sido suficiente para orientar su
vida.

Durante años Gus creyó firmemente algo muy simple:

**la tecnología era la solución a los problemas de la humanidad.**

Pero ahora comenzaba a sospechar que la tecnología también podía convertirse en otra cosa.

Una herramienta de concentración de poder.

Un sistema donde el conocimiento —que durante siglos había sido una forma de emancipación— comenzaba a ser 
absorbido por unas pocas élites tecnológicas.

La Pink Panther entendía que su amigo estaba en una transición intelectual.

Por eso aquella mañana decidió salir a caminar.

Terminó en el **núcleo de la Narvarte**, donde el olor de los comales siempre era irresistible.

Se pidió una **memela** generosa, con salsa y frijoles.

Demasiado generosa, de hecho.

============================================================
📄 Doc 2 (01.md)
============================================================
la miró curioso.

—¿Cuáles?

La Pink Panther respondió:

—Historias donde la inteligencia artificial no es el centro.
Historias donde lo importante es **la evolución de la mente humana**.

Y continuó:

—Tal vez tu siguiente paso no es competir con la inteligencia artificial…
sino pensar **cómo puede evolucionar la humanidad**.

Gus guardó silencio.

Ella siguió hablando.

—Si el sistema económico está empezando a tratar el conocimiento como algo desechable…
entonces quizá es momento de imaginar **otras formas de sociedad**.

—Otras filosofías.

—Otras maneras de entender al ser humano.

Gus se quedó pensando.

Recordó al adolescente que quería inventar la Psicohistoria.

Recordó todos los años en que creyó que la tecnología resolvería todo.

Y por primera vez en mucho tiempo, la pregunta que apareció en su mente no era sobre sistemas, ni modelos, ni 
arquitecturas.

Era una pregunta más grande.

============================================================
📄 Doc 3 (01.md)
============================================================
a artificial y arquitectura de sistemas. Ahora hablaba de civilizaciones, de cultura, de ética, de la naturaleza 
humana.

Y Pink Panther —que tenía un talento especial para observar— entendía por qué.

Gus estaba viendo algo que cada vez más gente comenzaba a percibir:
la inteligencia artificial avanzaba con velocidad extraordinaria, pero el discurso dominante giraba alrededor de 
una sola idea:

**automatizar todo.**

Automatizar el trabajo.
Automatizar el conocimiento.
Reducir la necesidad de humanos.

El conocimiento mismo parecía estar entrando a una especie de **industrialización masiva**.

Un día Gus mencionó una frase que había aprendido de uno de sus exalumnos más brillantes:

—Esto empieza a parecer **colonialismo de datos**.

La Pink Panther se quedó pensando en esa frase.

Porque conocía bien la historia de Gus.

============================================================
📄 Doc 4 (11.md)
============================================================
de un fragmento de El problema de los tres cuerpos, un libro donde una civilización extraterrestre construía una 
computadora digital utilizando seres vivos. Aunque el contexto era diferente, la Pink Panther había adaptado la 
idea para enseñar los fundamentos de los sistemas operativos de una manera más humana y práctica.
Con la simulación terminada y los estudiantes satisfechos, la Pink Panther se retiró, reflexionando sobre cómo la 
creatividad y un enfoque práctico podían hacer que incluso los temas más abstractos cobraran vida. Sabía que, 
después de esta experiencia, sus estudiantes estarían mejor preparados para abordar el código fuente, porque no 
solo lo entenderían, sino que lo habrían vivido.

============================================================
📄 Doc 5 (06

[Step 4: Duration 3.08 seconds| Input tokens: 18,417 | Output tokens: 380]

Reached max steps.

[Step 5: Duration 6.74 seconds| Input tokens: 24,542 | Output tokens: 830]


💡 RESPUESTA:
Basándome en la información encontrada en los documentos, la relación entre Pink Panther y Gus es de **amistad intelectual y mentoría filosófica**.

## La relación se caracteriza por:

**1. Amistad profunda:** Pink Panther se refiere a Gus como "su amigo" y muestra un interés genuino por su evolución intelectual.

**2. Observadora y consejera:** Pink Panther tiene "un talento especial para observar" y entiende los procesos de pensamiento de Gus, actuando como una guía en sus reflexiones.

**3. Diálogos sobre tecnología y humanidad:** Mantienen conversaciones profundas sobre:
- El impacto de la inteligencia artificial en la sociedad
- La evolución de la mente humana
- Nuevas formas de organización social
- La filosofía de la tecnología

**4. Contexto de transición intelectual:** Gus está experimentando un cambio en su perspectiva. Anteriormente creía que "la tecnología era la solución a los problemas de la humanidad", pero ahora cuestiona si también puede ser "una herramie

## 8. Chat interactivo con Gradio

In [ ]:
import gradio as gr


def chat(message: str, history: list) -> str:
    try:
        return str(agent.run(message))
    except Exception as e:
        return f"❌ Error: {str(e)}"


gr.ChatInterface(
    fn=chat,
    title="🐾 Pink Panther Knowledge Base - Chat RAG",
    description="Chatea con un agente que conoce las historias de Pink Panther.",
    examples=[
        "¿Qué piensa Pink Panther sobre la inteligencia artificial?",
        "¿Cuál es la relación entre Pink Panther y Gus?",
        "¿Qué reflexiones hay sobre el amor y la música?",
    ],
).launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b5e8b70a9026f4433d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ que opinas de Gustav Mahler ?                                                                                   │
│                                                                                                                 │
╰─ LiteLLMModel - bedrock/converse/us.anthropic.claude-sonnet-4-20250514-v1:0 ────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: The user is asking for my opinion about Gustav Mahler in Spanish. I need to search for information about  
Gustav Mahler in the knowledge base to provide a comprehensive response about this composer.                       
                                                                                                                   
<code>                                                                                                             
mahler_info = knowledge_base_search("Gustav Mahler compositor música sinfónica")                                   
print(mahler_info)                                                                                                 
                                                                                                                   

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: Based on the stories in the knowledge base, I can see that Gustav Mahler's music has a profound impact on 
Pink Panther, particularly his symphonies. Let me provide a comprehensive opinion about Gustav Mahler based on     
these experiences and musical insights.                                                                            
                                                                                                                   
<code>                                                                                                             
final_answer("""Gustav Mahler es, sin duda, uno de los compositores más extraordinarios y emotivamente poderosos de
la música clásica. Basándome en las experiencias que he encontrado en estas historias, mi opinión sobre Mahler es  
profundamente admirativa:                                                                                          
                                                                                                                   
**Como compositor emocional:** Mahler tiene una capacidad única para traducir las emociones humanas más profundas  
en música. Sus sinfonías no son solo obras técnicas, sino verdaderos viajes espirituales. La Sexta Sinfonía, por   
ejemplo, es descrita como una obra que "arrastra sin compasión" - una experiencia musical que toca las fibras más  
íntimas del alma.                                                                                                  
                                                                                                                   
**Su genio narrativo:** Cada sinfonía de Mahler cuenta una historia. La Segunda "Resurrección" habla de muerte y   
renacimiento, la Primera "Titán" captura el despertar de la naturaleza, y la Quinta expresa "la fuerza luminosa del
amor" por su esposa Alma. Mahler no compose música abstracta; compose vida.                                        
                                                                                                                   
**Su profundidad psicológica:** Lo que más me impresiona de Mahler es [38;2;230;237;243;48;2;13;17;23

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("""Gustav Mahler es, sin duda, uno de los compositores más extraordinarios y emotivamente           
  poderosos de la música clásica. Basándome en las experiencias que he encontrado en estas historias, mi opinión   
  sobre Mahler es profundamente admirativa:                                                                        
                                                                                                                   
  **Como compositor emocional:** Mahler tiene una capacidad única para traducir las emociones humanas más          
  profundas en música. Sus sinfonías no son solo obras técnicas, sino verdaderos viajes espirituales. La Sexta     
  Sinfonía, por ejemplo, es descrita como una obra que "arrastra sin compasión" - una experiencia musical que      
  toca las fibras más íntimas del alma.                                                                            
                                                                                                                   
  **Su genio narrativo:** Cada sinfonía de Mahler cuenta una historia. La Segunda "Resurrección" habla de muerte   
  y renacimiento, la Primera "Titán" captura el despertar de la naturaleza, y la Quinta expresa "la fuerza         
  luminosa del amor" por su esposa Alma. Mahler no compose música abstracta; compose vida.                         
                                                                                                                   
  **Su profundidad psicológica:** Lo que más me impresiona de Mahler es cómo logra crear un "diálogo vivo entre    
  la ternura y la brutalidad del mundo". Sus obras reflejan la complejidad del espíritu humano - el amor, el       
  dolor, la esperanza, la desesperación - todo convive en sus partituras.                                          
                                                                                                                   
  **Su legado transformador:** La música de Mahler tiene el poder de cambiar perspectivas y renovar el espíritu.   
  Como se describe en las historias, después de escuchar sus sinfonías, uno sale "armado con una nueva             
  perspectiva sobre la vida".                                                                                      
                                                                                                                   
  En resumen, considero a Gustav Mahler no solo como un gran compositor, sino como un poeta musical que logró      
  capturar la esencia misma de la experiencia humana y transformarla en arte eterno.""")                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Gustav Mahler es, sin duda, uno de los compositores más extraordinarios y emotivamente poderosos de 
la música clásica. Basándome en las experiencias que he encontrado en estas historias, mi opinión sobre Mahler es 
profundamente admirativa:

**Como compositor emocional:** Mahler tiene una capacidad única para traducir las emociones humanas más profundas 
en música. Sus sinfonías no son solo obras técnicas, sino verdaderos viajes espirituales. La Sexta Sinfonía, por 
ejemplo, es descrita como una obra que "arrastra sin compasión" - una experiencia musical que toca las fibras más 
íntimas del alma.

**Su genio narrativo:** Cada sinfonía de Mahler cuenta una historia. La Segunda "Resurrección" habla de muerte y 
renacimiento, la Primera "Titán" captura el despertar de la naturaleza, y la Quinta expresa "la fuerza luminosa del
amor" por su esposa Alma. Mahler no compose música abstracta; compose vida.

**Su profundidad psicológica:** Lo que más me impresiona de Mahler es cómo logra crear un "diálogo vivo entre la 
ternura y la brutalidad del mundo". Sus obras reflejan la complejidad del espíritu humano - el amor, el dolor, la 
esperanza, la desesperación - todo convive en sus partituras.

**Su legado transformador:** La música de Mahler tiene el poder de cambiar perspectivas y renovar el espíritu. Como
se describe en las historias, después de escuchar sus sinfonías, uno sale "armado con una nueva perspectiva sobre 
la vida".

En resumen, considero a Gustav Mahler no solo como un gran compositor, sino como un poeta musical que logró 
capturar la esencia misma de la experiencia humana y transformarla en arte eterno.

[Step 3: Duration 8.83 seconds| Input tokens: 12,197 | Output tokens: 705]

## 9. Pregunta libre

Cambia la pregunta y ejecuta la celda.

In [ ]:
mi_pregunta = "¿Qué enseña Pink Panther sobre la música?"

print(agent.run(mi_pregunta))